# 05 — EventWarpNet V2 (Two-Stage Flow + Event Attention)
Two key improvements over the base EventWarpNet:

**1. Two-Stage Flow (TimeLens-inspired):**
- Stage 1: Events-only → coarse bidirectional flow (events ARE the motion signal)
- Stage 2: RGB + coarse warp → residual flow refinement (RGB resolves ambiguities)

**2. Event Attention Gates in RefineNet:**
- Events generate spatial attention at each decoder level
- Focuses refinement capacity on motion regions where warping needs correction
- Static regions pass through with minimal processing

In [ ]:
# ── Cell 1: Setup & Mount ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/493Project')

%pip install -q torchmetrics

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 26.6 MB/s eta 0:00:00


In [ ]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import importlib
import src.models, src.train, src.losses, src.data
importlib.reload(src.models)
importlib.reload(src.losses)
importlib.reload(src.train)
importlib.reload(src.data)

import os
import glob
import random
import torch
from torch.amp import GradScaler
from torch.utils.data import DataLoader

from src.data import VimeoTripletDataset
from src.train import (
    build_model,
    build_criterion,
    build_optimizer,
    build_scheduler,
    load_checkpoint,
    overfit_one_batch,
    run_training,
)
from src.losses import CharbonnierLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────────────────
# Fine-tune from v8 checkpoint with perceptual loss to sharpen residual
WARMSTART_CKPT = '/content/drive/MyDrive/493Project/checkpoints/vimeo_v8_warp_v2/best_model.pth'

cfg = dict(
    # Paths
    vimeo_root     = '/content/drive/MyDrive/493Project/data/vimeo',
    local_root     = '/content/local_vimeo',
    checkpoint_dir = '/content/drive/MyDrive/493Project/checkpoints/vimeo_v9_warp_v2_perc',

    # Model
    model_type     = 'warp_v2',

    # Data
    patch_size     = 256,
    batch_size     = 8,
    num_workers    = 4,

    # Optimizer — lower LR for fine-tuning
    lr             = 5e-5,
    lr_min         = 1e-6,
    weight_decay   = 0.0,

    # Scheduler
    warmup_epochs  = 0,

    # Loss — add perceptual to wake up the residual
    lambda_char            = 1.0,
    lambda_perceptual      = 0.1,   # NEW — gives gradient signal for sharpness
    lambda_ssim            = 0.0,
    lambda_smooth          = 0.01,
    lambda_event_weighted  = 0.0,

    # Training
    num_epochs     = 30,
    max_norm       = 1.0,
)

os.makedirs(cfg['checkpoint_dir'], exist_ok=True)
print(f'Model: {cfg["model_type"]}')
print(f'Warm-start: {WARMSTART_CKPT}')
print(f'Loss: Charb({cfg["lambda_char"]}) + Perc({cfg["lambda_perceptual"]}) + Smooth({cfg["lambda_smooth"]})')
print(f'LR: {cfg["lr"]} | Epochs: {cfg["num_epochs"]}')

Model: warp_v2
Warm-start: /content/drive/MyDrive/493Project/checkpoints/vimeo_v8_warp_v2/best_model.pth
Loss: Charb(1.0) + Perc(0.1) + Smooth(0.01)
LR: 5e-05 | Epochs: 30


In [ ]:
# ── Cell 4: Load data ────────────────────────────────────────────────────────
import subprocess

vimeo_root = cfg['vimeo_root']
local_root = cfg['local_root']
os.makedirs(local_root, exist_ok=True)

tar_on_drive = os.path.join(vimeo_root, 'processed.tar')
marker = os.path.join(local_root, '.copy_done')

if not os.path.exists(marker):
    assert os.path.exists(tar_on_drive), \
        f'processed.tar not found on Drive — run 00b_tar_processed.ipynb first'
    subprocess.run(['apt-get', 'install', '-qq', '-y', 'pv'], capture_output=True)
    total_bytes = os.path.getsize(tar_on_drive)
    print(f'Extracting processed.tar ({total_bytes / 1e9:.1f} GB) to local SSD...')
    subprocess.run(
        f'pv -f -s {total_bytes} "{tar_on_drive}" | tar xf - -C "{local_root}"',
        shell=True, check=True,
    )
    open(marker, 'w').close()
    print('Done.')
else:
    print('Local data already exists, skipping extraction.')

def load_split(split_file):
    path = os.path.join(vimeo_root, split_file)
    with open(path) as f:
        entries = [line.strip() for line in f if line.strip()]
    dirs = []
    for rel in entries:
        d = os.path.join(local_root, rel)
        if os.path.isdir(d):
            dirs.append((rel, d))
    return dirs

train_entries = load_split('tri_trainlist.txt')
val_entries   = load_split('tri_vallist.txt')

if len(val_entries) == 0 and len(train_entries) > 0:
    print(f'No processed val triplets — splitting 90/10 from {len(train_entries)} train entries')
    rng = random.Random(42)
    all_entries = list(train_entries)
    rng.shuffle(all_entries)
    n_val = max(1, int(len(all_entries) * 0.10))
    val_entries   = all_entries[:n_val]
    train_entries = all_entries[n_val:]

train_dirs = [d for _, d in train_entries]
val_dirs   = [d for _, d in val_entries]
print(f'Train: {len(train_dirs)} | Val: {len(val_dirs)}')

Extracting processed.tar (5.3 GB) to local SSD...
Done.
No processed val triplets — splitting 90/10 from 1900 train entries
Train: 1710 | Val: 190


In [ ]:
# ── Cell 5: Datasets and dataloaders ──────────────────────────────────────────
train_dataset = VimeoTripletDataset(train_dirs, patch_size=cfg['patch_size'], augment=True)
val_dataset   = VimeoTripletDataset(val_dirs,   patch_size=cfg['patch_size'], augment=False)

train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'], shuffle=True,
                          num_workers=cfg['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg['batch_size'], shuffle=False,
                          num_workers=cfg['num_workers'], pin_memory=True)

print(f'Train: {len(train_dataset)} clips, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} clips, {len(val_loader)} batches')

Train: 1710 clips, 214 batches
Val:   190 clips, 24 batches


In [ ]:
# ── Cell 6: Model + warm-start from Charb-only checkpoint ────────────────────
model = build_model(cfg, device)

# Load pre-trained weights (v8: Charb-only, good flow, weak residual)
import os as _os
if _os.path.exists(WARMSTART_CKPT):
    state_dict = torch.load(WARMSTART_CKPT, map_location=device)
    model.load_state_dict(state_dict)
    print(f'Loaded warm-start weights from {WARMSTART_CKPT}')
else:
    print(f'No warm-start checkpoint found at {WARMSTART_CKPT} — training from scratch')

# Fresh optimizer (don't resume optimizer state — loss landscape changed)
optimizer = build_optimizer(model, cfg)
scheduler = build_scheduler(optimizer, cfg)
scaler    = GradScaler('cuda', enabled=False)
criterion = build_criterion(cfg, device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: EventWarpNetV2 | Parameters: {n_params:,}')
print(f'LR: {cfg["lr"]} | AMP: {scaler.is_enabled()}')

Loaded warm-start weights from /content/drive/MyDrive/493Project/checkpoints/vimeo_v8_warp_v2/best_model.pth
Model: EventWarpNetV2 | Parameters: 15,085,292
LR: 5e-05 | AMP: False


In [ ]:
# ── Cell 7: Sanity check — overfit one batch ─────────────────────────────────
overfit_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                            num_workers=0, pin_memory=True)
sane_model     = build_model(cfg, device)
sane_optimizer = torch.optim.Adam(sane_model.parameters(), lr=2e-3)
sane_criterion = CharbonnierLoss()

final_loss = overfit_one_batch(
    sane_model, overfit_loader, sane_criterion, sane_optimizer, device, iters=500
)
print(f'Overfit final loss: {final_loss:.6f}  (target < 0.005)')
del sane_model, sane_optimizer, sane_criterion, overfit_loader

Overfitting batch: 100%|██████████| 500/500 [02:08<00:00,  3.88it/s, loss=0.025486]

Overfit final loss: 0.025486  (target < 0.005)


In [ ]:
 # ── Cell 8: Full training loop ───────────────────────────────────────────────
training_log = run_training(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler, scaler,
    device, cfg,
    start_epoch=1,
    best_psnr=0.0,
)
print('Training complete.')

Epoch 1: 100%|██████████| 214/214 [02:09<00:00,  1.65it/s, loss=0.0599, gnorm=0.8]


Epoch 001/30 | Train 0.0578 | Val 0.0584 | PSNR 31.07 dB | SSIM 0.9309
  -> New best PSNR: 31.07 dB


Epoch 2: 100%|██████████| 214/214 [02:09<00:00,  1.65it/s, loss=0.0716, gnorm=1.0]


Epoch 002/30 | Train 0.0572 | Val 0.0571 | PSNR 31.27 dB | SSIM 0.9326
  -> New best PSNR: 31.27 dB


Epoch 3: 100%|██████████| 214/214 [02:09<00:00,  1.65it/s, loss=0.0539, gnorm=0.8]


...

Epoch 028/30 | Train 0.0507 | Val 0.0515 | PSNR 31.76 dB | SSIM 0.9414
  -> New best PSNR: 31.76 dB


Epoch 29: 100%|██████████| 214/214 [02:09<00:00,  1.65it/s, loss=0.0455, gnorm=0.5]


Epoch 029/30 | Train 0.0507 | Val 0.0516 | PSNR 31.63 dB | SSIM 0.9413


Epoch 30: 100%|██████████| 214/214 [02:09<00:00,  1.65it/s, loss=0.0650, gnorm=0.4]


Epoch 030/30 | Train 0.0509 | Val 0.0516 | PSNR 31.74 dB | SSIM 0.9415
Training complete.


In [ ]:
# ── Cell 9: Qualitative results + flow visualization ─────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from PIL import Image
from src.train import _forward

best_model = build_model(cfg, device)
best_model.load_state_dict(
    torch.load(os.path.join(cfg['checkpoint_dir'], 'best_model.pth'), map_location=device)
)
best_model.eval()

def load_vimeo_clip(clip_dir):
    f0  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im1.png')).convert('RGB'))
    f1  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im3.png')).convert('RGB'))
    gt  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im2.png')).convert('RGB'))
    evt = torch.load(os.path.join(clip_dir, 'voxel.pt'), weights_only=True)
    return f0, f1, gt, evt

rng = random.Random(42)
show_dirs = rng.sample(val_dirs, min(6, len(val_dirs)))

cols = ['Frame f0', 'Ground Truth', 'Prediction', 'Error (x5)', 'Flow t→0 (mag)', 'Events (sum)']
fig, axes = plt.subplots(len(show_dirs), 6, figsize=(28, 4.5 * len(show_dirs)))
if len(show_dirs) == 1:
    axes = axes[np.newaxis, :]

for col, title in enumerate(cols):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold')

with torch.no_grad():
    for row, clip_dir in enumerate(show_dirs):
        f0, f1, gt, evt = load_vimeo_clip(clip_dir)
        _, H, W = f0.shape
        pad_h = (16 - H % 16) % 16
        pad_w = (16 - W % 16) % 16
        f0_d = torch.nn.functional.pad(f0, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)
        f1_d = torch.nn.functional.pad(f1, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)
        evt_d = torch.nn.functional.pad(evt, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)

        outputs = _forward(best_model, f0_d, f1_d, evt_d)
        pred = outputs[0].squeeze(0).float().cpu().clamp(0, 1)[:, :H, :W]
        flow_t0 = outputs[1].squeeze(0).float().cpu()[:, :H, :W]

        error = (gt - pred).abs() * 5
        mse = ((gt - pred) ** 2).mean().item()
        psnr = -10 * np.log10(mse + 1e-10)

        # Flow magnitude
        flow_mag = flow_t0.norm(dim=0).numpy()
        # Event sum
        evt_sum = evt.sum(dim=0).numpy()
        abs_max = max(abs(evt_sum.min()), abs(evt_sum.max()), 1e-8)

        axes[row, 0].imshow(f0.permute(1,2,0).numpy())
        axes[row, 1].imshow(gt.permute(1,2,0).numpy())
        axes[row, 2].imshow(pred.permute(1,2,0).clamp(0,1).numpy())
        axes[row, 3].imshow(error.permute(1,2,0).clamp(0,1).numpy())
        axes[row, 4].imshow(flow_mag, cmap='hot')
        axes[row, 5].imshow(evt_sum, cmap='RdBu_r', vmin=-abs_max, vmax=abs_max)

        for c in range(6):
            axes[row, c].set_xticks([]); axes[row, c].set_yticks([])
        axes[row, 2].set_xlabel(f'PSNR: {psnr:.2f} dB', fontsize=9, color='steelblue')
        axes[row, 4].set_xlabel(f'max={flow_mag.max():.1f}px', fontsize=8)

plt.suptitle('EventWarpNet V2 — Two-Stage Flow + Event Attention', fontsize=15, y=1.01)
plt.tight_layout()
fig_path = os.path.join(cfg['checkpoint_dir'], 'qualitative_results.png')
plt.savefig(fig_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved to {fig_path}')

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ── Cell 10: 3-way GIF — GT vs DualEncoder vs EventWarpNetV2 ─────────────────
import numpy as np
from PIL import Image, ImageDraw
import torchvision.transforms.functional as TF
from src.train import _forward, build_model
from IPython.display import display, HTML
import base64

# Load DualEncoderUNet for comparison
dual_cfg = {**cfg, 'model_type': 'dual'}
dual_model = build_model(dual_cfg, device)
dual_ckpt = '/content/drive/MyDrive/493Project/checkpoints/vimeo_v6_dual/best_model.pth'
dual_model.load_state_dict(torch.load(dual_ckpt, map_location=device))
dual_model.eval()

warp_model = build_model(cfg, device)
warp_model.load_state_dict(
    torch.load(os.path.join(cfg['checkpoint_dir'], 'best_model.pth'), map_location=device)
)
warp_model.eval()

rng = random.Random(99)
gif_dirs = rng.sample(val_dirs, min(8, len(val_dirs)))

def to_pil(tensor):
    return Image.fromarray((tensor.permute(1, 2, 0).clamp(0, 1).numpy() * 255).astype(np.uint8))

for i, clip_dir in enumerate(gif_dirs):
    f0  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im1.png')).convert('RGB'))
    f1  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im3.png')).convert('RGB'))
    gt  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im2.png')).convert('RGB'))
    evt = torch.load(os.path.join(clip_dir, 'voxel.pt'), weights_only=True)

    _, H, W = f0.shape
    pad_h = (16 - H % 16) % 16
    pad_w = (16 - W % 16) % 16
    f0_d = torch.nn.functional.pad(f0, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)
    f1_d = torch.nn.functional.pad(f1, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)
    evt_d = torch.nn.functional.pad(evt, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)

    with torch.no_grad():
        pred_dual = _forward(dual_model, f0_d, f1_d, evt_d)[0].squeeze(0).float().cpu().clamp(0,1)[:,:H,:W]
        pred_warp = _forward(warp_model, f0_d, f1_d, evt_d)[0].squeeze(0).float().cpu().clamp(0,1)[:,:H,:W]

    gt_frames   = [to_pil(f0), to_pil(gt),        to_pil(f1)]
    dual_frames = [to_pil(f0), to_pil(pred_dual), to_pil(f1)]
    warp_frames = [to_pil(f0), to_pil(pred_warp), to_pil(f1)]

    w, h = gt_frames[0].size
    gap, label_h = 6, 22
    combined_frames = []
    labels = ['Ground Truth', 'DualEncoder (31dB)', 'WarpNetV2']
    for g, d, wp in zip(gt_frames, dual_frames, warp_frames):
        combined = Image.new('RGB', (w*3 + gap*2, h + label_h), (255,255,255))
        combined.paste(g, (0, label_h))
        combined.paste(d, (w+gap, label_h))
        combined.paste(wp, (w*2+gap*2, label_h))
        draw = ImageDraw.Draw(combined)
        for ci, lbl in enumerate(labels):
            draw.text((ci*(w+gap) + w//2 - len(lbl)*3, 3), lbl, fill=(0,0,0))
        combined_frames.append(combined)

    gif_path = os.path.join(cfg['checkpoint_dir'], f'comparison_3way_gif_{i}.gif')
    combined_frames[0].save(gif_path, save_all=True, append_images=combined_frames[1:], duration=400, loop=0)

    with open(gif_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(f'<p><b>Sample {i}</b></p><img src="data:image/gif;base64,{b64}" />'))

print(f'\nSaved {len(gif_dirs)} GIFs to {cfg["checkpoint_dir"]}')

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ── Cell 11: Diagnostic — warp quality, residual, alpha analysis ─────────────
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from PIL import Image
from src.models import backwarp

best_model = build_model(cfg, device)
best_model.load_state_dict(
    torch.load(os.path.join(cfg['checkpoint_dir'], 'best_model.pth'), map_location=device)
)
best_model.eval()

rng = random.Random(77)
diag_dirs = rng.sample(val_dirs, min(6, len(val_dirs)))

fig, axes = plt.subplots(len(diag_dirs), 8, figsize=(36, 4.5 * len(diag_dirs)))
if len(diag_dirs) == 1:
    axes = axes[np.newaxis, :]

titles = ['GT', 'Prediction', 'Warped f0', 'Warped f1', 'Alpha mask',
          'Residual (abs, x5)', 'Flow mag', 'Error (x5)']
for col, t in enumerate(titles):
    axes[0, col].set_title(t, fontsize=11, fontweight='bold')

residual_stats = []
alpha_stats = []
flow_stats = []

with torch.no_grad():
    for row, clip_dir in enumerate(diag_dirs):
        f0 = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im1.png')).convert('RGB'))
        f1 = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im3.png')).convert('RGB'))
        gt = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im2.png')).convert('RGB'))
        evt = torch.load(os.path.join(clip_dir, 'voxel.pt'), weights_only=True)

        _, H, W = f0.shape
        pad_h = (16 - H % 16) % 16
        pad_w = (16 - W % 16) % 16
        f0_d = torch.nn.functional.pad(f0, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)
        f1_d = torch.nn.functional.pad(f1, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)
        evt_d = torch.nn.functional.pad(evt, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)

        # Full forward to get all intermediates
        outputs = best_model(torch.cat([f0_d, f1_d], dim=1), evt_d)
        pred = outputs[0].squeeze(0).cpu().clamp(0, 1)[:, :H, :W]
        flow_t0 = outputs[1].squeeze(0).cpu()[:, :H, :W]
        flow_t1 = outputs[2].squeeze(0).cpu()[:, :H, :W]

        # Reconstruct warped frames and internals
        warped_f0 = backwarp(f0_d, outputs[1])[:, :, :H, :W].squeeze(0).cpu().clamp(0, 1)
        warped_f1 = backwarp(f1_d, outputs[2])[:, :, :H, :W].squeeze(0).cpu().clamp(0, 1)

        # Get alpha and residual from refinement
        rgb_in = torch.cat([f0_d, f1_d], dim=1)
        f0_m, f1_m = rgb_in[:, :3], rgb_in[:, 3:]
        ft0, ft1, _, _ = best_model.flownet(f0_m, f1_m, evt_d)
        wf0 = backwarp(f0_m, ft0)
        wf1 = backwarp(f1_m, ft1)
        ctx0 = backwarp(best_model.ctx_enc(f0_m), ft0)
        ctx1 = backwarp(best_model.ctx_enc(f1_m), ft1)
        refine_in = torch.cat([wf0, wf1, evt_d, ft0, ft1, ctx0, ctx1], dim=1)
        alpha, residual = best_model.refinenet(refine_in, evt_d)
        alpha = alpha.squeeze(0).cpu()[:, :H, :W]
        residual = residual.squeeze(0).cpu()[:, :H, :W]

        # Stats
        residual_stats.append({
            'mean': residual.abs().mean().item(),
            'std': residual.std().item(),
            'max': residual.abs().max().item(),
        })
        alpha_stats.append({
            'mean': alpha.mean().item(),
            'near_0_or_1': ((alpha < 0.1) | (alpha > 0.9)).float().mean().item(),
        })
        flow_mag = flow_t0.norm(dim=0)
        flow_stats.append({
            'mean': flow_mag.mean().item(),
            'max': flow_mag.max().item(),
        })

        error = (gt - pred).abs() * 5

        # Plot
        axes[row, 0].imshow(gt.permute(1,2,0).numpy())
        axes[row, 1].imshow(pred.permute(1,2,0).numpy())
        axes[row, 2].imshow(warped_f0.permute(1,2,0).numpy())
        axes[row, 3].imshow(warped_f1.permute(1,2,0).numpy())
        axes[row, 4].imshow(alpha.squeeze(0).numpy(), cmap='gray', vmin=0, vmax=1)
        axes[row, 5].imshow((residual.abs() * 5).permute(1,2,0).clamp(0,1).numpy())
        axes[row, 6].imshow(flow_mag.numpy(), cmap='hot')
        axes[row, 7].imshow(error.permute(1,2,0).clamp(0,1).numpy())

        mse = ((gt - pred)**2).mean().item()
        psnr = -10 * np.log10(mse + 1e-10)
        axes[row, 1].set_xlabel(f'{psnr:.1f} dB', fontsize=9, color='steelblue')
        axes[row, 4].set_xlabel(f'sharp: {alpha_stats[-1]["near_0_or_1"]*100:.0f}%', fontsize=8)
        axes[row, 5].set_xlabel(f'|res| max={residual_stats[-1]["max"]:.3f}', fontsize=8)
        axes[row, 6].set_xlabel(f'max={flow_stats[-1]["max"]:.1f}px', fontsize=8)

        for c in range(8):
            axes[row, c].set_xticks([]); axes[row, c].set_yticks([])

plt.suptitle('EventWarpNetV2 — Warp / Alpha / Residual Diagnostics', fontsize=14, y=1.01)
plt.tight_layout()
diag_path = os.path.join(cfg['checkpoint_dir'], 'warp_diagnostics.png')
plt.savefig(diag_path, bbox_inches='tight', dpi=150)
plt.show()

# Print summary
print('\n=== WARP QUALITY DIAGNOSTICS ===')
print(f'\nResidual stats (avg over {len(residual_stats)} clips):')
print(f'  Mean |residual|: {np.mean([r["mean"] for r in residual_stats]):.4f}')
print(f'  Residual std:    {np.mean([r["std"] for r in residual_stats]):.4f}')
print(f'  Max |residual|:  {np.mean([r["max"] for r in residual_stats]):.4f}')
print(f'\nAlpha mask stats:')
print(f'  Mean alpha:          {np.mean([a["mean"] for a in alpha_stats]):.3f}  (0.5 = equal blend, near 0 or 1 = decisive)')
print(f'  % decisive (>0.9 or <0.1): {np.mean([a["near_0_or_1"] for a in alpha_stats])*100:.1f}%')
print(f'\nFlow stats:')
print(f'  Mean flow magnitude: {np.mean([f["mean"] for f in flow_stats]):.2f} px')
print(f'  Max flow magnitude:  {np.mean([f["max"] for f in flow_stats]):.1f} px')
print(f'\n(Higher residual = model is correcting more warping artifacts)')
print(f'(Decisive alpha = sharp occlusion handling, not blurry blending)')
print(f'Saved to {diag_path}')

Output hidden; open in https://colab.research.google.com to view.